# HV-011 — Measurement Uncertainty Budget

**Technical gate:** MU-001  
**Dataset release:** VSM-LAB-DATA-v1.1  
**Public specification:** config/public/lab_spec_v1.json

This notebook builds a traceable uncertainty budget for a calibrated 60-second stable RH result at the four declared uncertainty-evaluation points.

## 1. Engineering Question

> What combined and expanded uncertainty is associated with a calibrated 60-second stable RH result, using only analyst-visible evidence and public laboratory specifications?

The notebook separates **residual bias** from **standard uncertainty** and explicitly prevents double-counting.

## 2. Reproducible Setup

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

current_dir = Path.cwd()
candidate_roots = [current_dir, *current_dir.parents]

project_root = next(
    (
        path
        for path in candidate_roots
        if (path / "data" / "raw" / "vsm_lab_data_v1_1").exists()
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError(
        "Could not locate project root containing "
        "'data/raw/vsm_lab_data_v1_1'."
    )

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.uncertainty_analysis import (
    build_sensitivity_table,
    build_uncertainty_budget,
    load_frozen_c1_offset,
    load_public_lab_spec,
    load_repeatability_sd,
    load_stability_excursion,
    load_static_and_matrix,
    point_evidence_table,
)

tables_dir = project_root / "reports" / "tables"
figures_dir = project_root / "reports" / "figures"
tables_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)

print(f"Project root: {project_root}")

## 3. MU-01 — Public Specification and Provenance Audit

The downstream repository contains an imported analyst-visible public specification with upstream provenance.

No private simulator truth is used.

In [ ]:
public_spec = load_public_lab_spec(project_root)

ref_spec = public_spec["reference_public_spec"]
dut_spec = public_spec["dut_public_spec"]
chamber_spec = public_spec["chamber_public_spec"]
daq_spec = public_spec["daq_public_spec"]
unc_points = public_spec["experimental_design"]["uncertainty_evaluation_points"]

assert public_spec["lab_identity"]["measurand"] == "relative_humidity"
assert public_spec["lab_identity"]["units"] == "%RH"
assert len(unc_points) == 4
assert np.isclose(
    ref_spec["expanded_calibration_uncertainty_rh_percent"],
    0.40,
)
assert np.isclose(ref_spec["coverage_factor_k"], 2.0)

print("Declared uncertainty-evaluation points")
display(pd.DataFrame(unc_points))

print("Reference expanded calibration uncertainty:",
      ref_spec["expanded_calibration_uncertainty_rh_percent"], "%RH")
print("Reference coverage factor:",
      ref_spec["coverage_factor_k"])

print("MU-01 public specification audit: PASS")

## 4. MU-02 — Frozen Measurement Model

Primary reported result:

```text
Y = mean(RH_DUT over 60 s) + C1
```

Frozen C1:

```text
+1.2266788194 %RH
```

Dynamic measurements are outside this uncertainty model.

In [ ]:
frozen_c1_offset = load_frozen_c1_offset(project_root)
repeatability_sd = load_repeatability_sd(project_root)
stability_excursion = load_stability_excursion(project_root)

u_repeat_mean = repeatability_sd / np.sqrt(60.0)
u_ref_cal = (
    ref_spec["expanded_calibration_uncertainty_rh_percent"]
    / ref_spec["coverage_factor_k"]
)
u_stability = stability_excursion / np.sqrt(3.0)

assert np.isclose(frozen_c1_offset, 1.2266788194444438)

print(f"Frozen C1 offset        : {frozen_c1_offset:.10f} %RH")
print(f"Pooled repeatability SD : {repeatability_sd:.6f} %RH")
print(f"u_repeat_mean           : {u_repeat_mean:.6f} %RH")
print(f"u_ref_cal               : {u_ref_cal:.6f} %RH")
print(f"Stability excursion     : {stability_excursion:.6f} %RH")
print(f"u_stability             : {u_stability:.6f} %RH")

print("MU-02 frozen measurement model: PASS")

## 5. MU-03 — Point-Specific Residual/State Evidence

Evidence role is frozen as:

- 30/25, 50/25, 70/25 → independent **VALIDATION** data;
- 80/35 → independent **CHARACTERIZATION** data.

Each point must contain four Test_IDs corresponding to Run × Direction structure.

Point-level state variability removes the repeatability-of-mean contribution before combination:

```text
u_state = sqrt(max(s_point² - u_repeat_mean², 0))
```

In [ ]:
static_df, test_matrix_df = load_static_and_matrix(project_root)

point_test_evidence_df, point_summary_df = point_evidence_table(
    static_df=static_df,
    test_matrix_df=test_matrix_df,
    public_spec=public_spec,
    c1_offset=frozen_c1_offset,
    repeatability_sd=repeatability_sd,
    samples_per_result=60,
)

assert len(point_summary_df) == 4
assert (point_summary_df["N_Test_IDs"] == 4).all()

print("Test_ID-level evidence")
display(point_test_evidence_df.round(5))

print("Point-level evidence")
display(point_summary_df.round(5))

print("MU-03 point-specific evidence: PASS")

## 6. MU-04 — Primary Uncertainty Budget

Primary standard components:

1. repeatability of the 60-s mean;
2. point-specific residual/state variability;
3. reference calibration uncertainty;
4. bounded observed stability excursion.

Residual bias remains separate.

In [ ]:
budget_df, uncertainty_summary_df = build_uncertainty_budget(
    point_summary_df=point_summary_df,
    public_spec=public_spec,
    stability_excursion=stability_excursion,
    coverage_factor=2.0,
)

display(budget_df.round(6))
print("Point uncertainty summary")
display(
    uncertainty_summary_df[
        [
            "Point_ID",
            "RH_Percent",
            "Temperature_C",
            "Evidence_Role",
            "Point_Residual_Bias",
            "Point_MBE_SD",
            "U_State",
            "U_Reference_Calibration",
            "U_Stability",
            "Combined_Standard_Uncertainty",
            "Expanded_Uncertainty",
            "Abs_Bias_Plus_Expanded_Uncertainty",
        ]
    ].round(5)
)

assert np.isfinite(
    uncertainty_summary_df["Combined_Standard_Uncertainty"]
).all()
assert np.isfinite(
    uncertainty_summary_df["Expanded_Uncertainty"]
).all()

print("MU-04 primary uncertainty budget: PASS")

## 7. MU-05 — Double-Counting Audit

The following are deliberately **not** added as separate primary components:

- DUT resolution / DUT quantization;
- reference resolution / reference quantization;
- reference repeatability;
- HV-005 Run and Day paired shifts;
- HV-010 short-term stability scatter;
- public reference drift limit;
- HV-010 span/shape change.

Reasoning:

- quantization and short-term scatter are already present in empirical repeatability;
- point-specific between-Test_ID variability captures state/run/direction variation;
- observed stability drift already contains analyst-visible DUT-reference temporal behavior;
- span/shape change comes from the same aging campaign and is dependent on that evidence.

In [ ]:
excluded_primary_components = pd.DataFrame(
    [
        ["DUT resolution / quantization", "Embedded in empirical repeatability"],
        ["Reference resolution / quantization", "Embedded in empirical repeatability"],
        ["Reference repeatability", "Embedded in empirical DUT-reference repeatability"],
        ["HV-005 Run/Day shifts", "State/systematic structure; avoid duplication with point variability"],
        ["HV-010 short-term scatter", "Overlaps repeatability"],
        ["Reference drift limit", "Sensitivity only; may overlap observed stability drift"],
        ["HV-010 span/shape change", "Dependent stability evidence; carry as documented condition"],
    ],
    columns=["Candidate_Component", "Primary_Treatment"],
)

display(excluded_primary_components)
print("MU-05 double-counting audit: PASS")

## 8. MU-06 — Sensitivity Cases

Two non-primary sensitivity cases are reported:

### Reference drift

Public annual drift limit:

```text
0.15 %RH/year
```

scaled to 90 days and modeled only for sensitivity.

### Chamber uniformity

Public chamber RH uniformity:

```text
±0.30 %RH
```

The public specification does not define DUT/reference placement or a differential spatial model, so this is not promoted to the primary budget.

In [ ]:
sensitivity_df = build_sensitivity_table(
    uncertainty_summary_df,
    public_spec,
)

display(
    sensitivity_df[
        [
            "Point_ID",
            "Case",
            "Combined_Standard_Uncertainty",
            "Expanded_Uncertainty_k2",
            "Reference_Drift_90d_Bound",
            "U_Reference_Drift_90d",
            "Chamber_Uniformity_Bound",
            "U_Chamber_Uniformity_Sensitivity",
        ]
    ].round(6)
)

print("MU-06 sensitivity analysis: PASS")

## 9. MU-07 — Variance Contribution Diagnostics

The budget is examined in variance space so the dominant uncertainty sources are visible.

In [ ]:
variance_summary_df = budget_df.copy()
variance_summary_df["Total_Point_Variance"] = (
    variance_summary_df
    .groupby("Point_ID")["Variance_Contribution"]
    .transform("sum")
)
variance_summary_df["Variance_Fraction"] = (
    variance_summary_df["Variance_Contribution"]
    / variance_summary_df["Total_Point_Variance"]
)

display(
    variance_summary_df[
        [
            "Point_ID",
            "Component",
            "Standard_Uncertainty",
            "Variance_Fraction",
        ]
    ].round(5)
)

print("MU-07 variance contribution diagnostics: PASS")

## 10. Curated Figures

In [ ]:
# Component standard uncertainty by point
component_pivot = budget_df.pivot(
    index="Point_ID",
    columns="Component",
    values="Standard_Uncertainty",
)

fig, ax = plt.subplots(figsize=(9, 5))
component_pivot.plot(kind="bar", ax=ax)
ax.set_xlabel("Evaluation point")
ax.set_ylabel("Standard uncertainty (%RH)")
ax.set_title("HV-011 Standard Uncertainty Components")
ax.legend(title="Component")
fig.tight_layout()
fig.savefig(
    figures_dir / "hv011_uncertainty_components.png",
    dpi=160,
    bbox_inches="tight",
)
plt.show()

# Expanded uncertainty by point
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(
    uncertainty_summary_df["Point_ID"],
    uncertainty_summary_df["Expanded_Uncertainty"],
)
ax.set_xlabel("Evaluation point")
ax.set_ylabel("Expanded uncertainty U, k=2 (%RH)")
ax.set_title("HV-011 Expanded Uncertainty by Evaluation Point")
fig.tight_layout()
fig.savefig(
    figures_dir / "hv011_expanded_uncertainty_by_point.png",
    dpi=160,
    bbox_inches="tight",
)
plt.show()

print("Curated HV-011 figures exported.")

## 11. Curated Evidence Export

In [ ]:
point_test_evidence_df.to_csv(
    tables_dir / "hv011_uncertainty_point_evidence.csv",
    index=False,
)
budget_df.to_csv(
    tables_dir / "hv011_uncertainty_budget.csv",
    index=False,
)
uncertainty_summary_df.to_csv(
    tables_dir / "hv011_uncertainty_summary.csv",
    index=False,
)
sensitivity_df.to_csv(
    tables_dir / "hv011_uncertainty_sensitivity.csv",
    index=False,
)

print("Curated HV-011 tables exported.")

## 12. Initial MU-001 Checkpoint

A clean-kernel Run All should establish:

- MU-01 public specification audit PASS;
- MU-02 frozen measurement model PASS;
- MU-03 point-specific evidence PASS;
- MU-04 primary uncertainty budget PASS;
- MU-05 double-counting audit PASS;
- MU-06 sensitivity analysis PASS;
- MU-07 variance contribution diagnostics PASS;
- four curated CSV tables;
- two curated figures.

The next step is engineering interpretation and formal MU-001 gate closure.

## 13. Engineering Interpretation

### 13.1 Primary expanded uncertainty is approximately 1.18–1.23 %RH

The primary k=2 expanded uncertainties are:

```text
UNC-001 — 30 %RH / 25 °C: U = 1.2297 %RH
UNC-002 — 50 %RH / 25 °C: U = 1.1823 %RH
UNC-003 — 70 %RH / 25 °C: U = 1.1831 %RH
UNC-004 — 80 %RH / 35 °C: U = 1.2216 %RH
```

The combined standard uncertainty is therefore approximately:

```text
0.591–0.615 %RH
```

across the four declared evaluation points.

The relatively narrow range indicates that the uncertainty magnitude is driven more by common experimental/state effects than by RH level itself.

### 13.2 Residual bias remains separate from uncertainty

Point residual bias after frozen C1:

```text
30 %RH / 25 °C: +0.0892 %RH
50 %RH / 25 °C: +0.2271 %RH
70 %RH / 25 °C: +0.1720 %RH
80 %RH / 35 °C: -0.1577 %RH
```

These biases are not folded into the standard uncertainty.

A conservative bias-plus-expanded-uncertainty envelope is:

```text
UNC-001: |b| + U ≈ 1.3189 %RH
UNC-002: |b| + U ≈ 1.4094 %RH
UNC-003: |b| + U ≈ 1.3551 %RH
UNC-004: |b| + U ≈ 1.3794 %RH
```

This preserves the distinction between systematic residual error and uncertainty around the reported result.

### 13.3 Point-state residual variability dominates the budget

Variance contribution by source is approximately:

```text
Point-state residual:        64–67 %
Stability bounded excursion: 22–24 %
Reference calibration:       10–11 %
Repeatability of 60-s mean:  ~0.3 %
```

The dominant component is therefore not short-term noise.

It is the observable between-Test_ID state structure, driven mainly by Direction and secondarily by Run/state differences.

This is consistent with earlier milestones:

- HV-006 and HV-007 found persistent Direction dependence;
- HV-009 showed strong direction-dependent dynamic behavior;
- HV-010 showed measurable temporal excursions.

The uncertainty budget therefore preserves the project's central physical finding: the DUT is highly repeatable within a short hold, but its reported value depends materially on measurement state/history.

### 13.4 Averaging 60 samples reduces only the small repeatability component

The pooled short-term repeatability SD is:

```text
s_repeat ≈ 0.2593 %RH
```

For a 60-s mean:

```text
u_repeat_mean ≈ 0.03347 %RH
```

This contributes only about 0.3% of total variance.

Therefore longer averaging alone would provide very little reduction in the total uncertainty unless the dominant state/direction and stability effects are also controlled.

### 13.5 Reference calibration uncertainty is meaningful but not dominant

The public reference specification gives:

```text
U_ref = 0.40 %RH, k = 2
u_ref = 0.20 %RH
```

This contributes approximately 10–11% of total variance.

Thus the reference instrument is sufficiently important that it cannot be treated as exact truth, but improving the reference calibration alone would not remove the dominant uncertainty source.

### 13.6 Stability treatment is intentionally conservative

HV-010 observed a maximum Day-0-referenced center excursion of:

```text
0.5025 %RH
```

The primary budget treats this as a bounded rectangular effect:

```text
u_stability = 0.5025 / sqrt(3)
            ≈ 0.2901 %RH
```

This contributes approximately 22–24% of total variance.

This treatment is conservative and transparent. It does not claim that the observed non-monotonic drift process is actually uniformly distributed.

### 13.7 Sensitivity cases do not change the engineering picture

Reference-drift sensitivity:

```text
90-day reference drift bound ≈ 0.0370 %RH
u_ref_drift ≈ 0.0214 %RH
```

Adding it changes expanded uncertainty by less than about 0.001 %RH.

Therefore separate reference-drift treatment is negligible compared with the primary budget and would risk double-counting observed stability behavior.

Chamber-uniformity sensitivity:

```text
±0.30 %RH public bound
u_chamber_sensitivity ≈ 0.1732 %RH
```

If conservatively added, expanded uncertainty rises to approximately:

```text
1.232–1.278 %RH
```

The result is not highly sensitive to this assumption.

However, chamber uniformity remains a sensitivity case because the public specification does not identify DUT/reference spatial placement or a defensible differential spatial model.

### 13.8 Static uncertainty-aware comparison with the public ±2 %RH accuracy limit

The public DUT specification declares:

```text
static accuracy limit = ±2.0 %RH
```

At all four declared uncertainty-evaluation points:

```text
|residual bias| + U < 2.0 %RH
```

even under the more conservative chamber-uniformity sensitivity case.

Therefore the available **static, 60-s stable-result evidence** is compatible with the public ±2 %RH accuracy requirement at these four points when a conservative k=2 uncertainty allowance is included.

This is not the project's final suitability decision.

Dynamic response, hysteresis, drift limits, and other public requirements remain separate compliance questions for the final engineering assessment.

## 14. MU-001 Gate Decision

**MU-001 = PASS WITH DOCUMENTED CONDITIONS**

### Basis for PASS

- the public laboratory specification is traceably imported from the upstream analyst-visible source;
- all four declared uncertainty-evaluation points are analyzed;
- frozen C1 remains unchanged;
- the reported measurand is explicitly defined as a calibrated 60-s stable mean;
- repeatability-of-mean is derived from HV-005;
- point-specific state variability is estimated without double-counting repeatability-of-mean;
- reference calibration uncertainty is included from public metrological specification;
- stability excursion is explicitly modeled as a bounded component;
- residual bias remains separate from uncertainty;
- sensitivity cases are clearly separated from the primary budget;
- no hidden simulator truth is used.

### Documented conditions

1. **Dominant state dependence:** approximately 64–67% of budget variance comes from point-specific between-Test_ID state variability, principally Direction/Run structure.
2. **Conservative stability model:** the 0.5025 %RH observed stability excursion is represented using a rectangular bound; this is a modeling choice, not an empirically demonstrated probability distribution.
3. **Limited point replication:** each evaluation point has four independent Test_ID means.
4. **80 %RH / 35 °C evidence role:** UNC-004 uses independent CHARACTERIZATION evidence because the blind VALIDATION design does not include 80 %RH.
5. **Chamber geometry unavailable:** chamber uniformity is sensitivity-only because public DUT/reference spatial placement is unspecified.
6. **Bias remains explicit:** residual bias is not converted into random uncertainty or post-validation correction.
7. **Static scope:** this uncertainty budget applies to a 60-s stable result and does not cover transient/dynamic tracking error.

### Primary result

Across the four declared evaluation points:

```text
u_c ≈ 0.591–0.615 %RH
U(k=2) ≈ 1.182–1.230 %RH
```

The next engineering stage is:

```text
HV-012 — Monte Carlo Uncertainty
```